# Explore built_subset

Quick look at the output of `02_build.py`. Everything here is real patient data: keep the notebook and its output on Bianca.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 100, "display.max_columns", 50, "display.width", 200)
BUILT = Path("/home/theon/Desktop/data_std/built_subset")

t = {f.stem: pd.read_parquet(f) for f in sorted(BUILT.glob("*.parquet"))}
for name, df in t.items():
    print(f"{name:15s} {len(df):>9,} rows  {df.shape[1]:>3} cols")

## Columns, types and missing values

In [ ]:
for name, df in t.items():
    print(f"\n=== {name} ===")
    display(pd.DataFrame({"dtype": df.dtypes.astype(str), "null_share": df.isna().mean().round(3)}))
    display(df.head(3))

## Build report and sanity checks

In [ ]:
display(pd.read_csv(BUILT / "build_report.csv"))

TOLVAN = 701403
for name, df in t.items():
    if "lopnr_swed" in df and (df["lopnr_swed"] == TOLVAN).any():
        print(f"WARNING: Tolvan is still in {name}")
print("ed_visit_id unique:", t["ed_visits"]["ed_visit_id"].is_unique)
print("one death record per person:", t["deaths"]["lopnr_swed"].is_unique)

## Patients and ED visits

In [ ]:
p, ed = t["patients"], t["ed_visits"]
print("patients:", len(p), "| share male:", round(pd.to_numeric(p["is_male"], errors="coerce").mean(), 3), "| share died:", round(p["date_death_minimp"].notna().mean(), 3))
display(p["n_ed_visits"].value_counts().sort_index().head(15))
display(ed.groupby("ed_hospital").agg(visits=("ed_visit_id", "size"), patients=("lopnr_swed", "nunique"),
                                      age_median=("ed_age", "median"), first=("ed_arrival_datetime", "min"),
                                      last=("ed_arrival_datetime", "max")))
display(ed.groupby("ed_hospital")[["ed_time_to_doctor_minutes", "ed_time_total_minutes"]].median())

## Vital signs

In [ ]:
v = t["vitals"]
display(pd.crosstab(v["concept"], v["source"]))
# Units per hospital (the supervisor's question): how each unit was decided
display(pd.crosstab([v["source"], v["concept"]], v["unit_status"]))
display(v[~v["implausible"]].groupby("concept")["value"].describe(percentiles=[.05, .5, .95]).round(1))
print("share implausible:", round(v["implausible"].mean(), 4), "| share linked to a visit:", round(v["ed_visit_id"].notna().mean(), 3))

In [ ]:
# Which term_names became which concept (check the regex mapping)
display(v.groupby(["concept", "source", "term_name"]).size().rename("n").reset_index())
# Blood pressure that could not be split (e.g. VGR without value_str)
display(v[v["concept"] == "bp_combined"].groupby("source").size())

In [ ]:
ok = v[~v["implausible"]]
concepts = sorted(ok["concept"].unique())
rows = (len(concepts) + 3) // 4
fig, axes = plt.subplots(rows, 4, figsize=(16, 3 * rows), squeeze=False)
for ax, c in zip(axes.flat, concepts):
    ok.loc[ok["concept"] == c, "value"].hist(bins=50, ax=ax)
    ax.set_title(c)
plt.tight_layout()

## Labs

In [ ]:
lab = t["lab_meas"]
print("share with SNOMED:", round(lab["snomed_id"].notna().mean(), 3), "| share linked to a visit:", round(lab["ed_visit_id"].notna().mean(), 3))
display(pd.crosstab(lab["source"], lab["unit_status"]))
display(lab.groupby(["source", "term_name", "unit"]).agg(n=("value_num", "size"), median=("value_num", "median"))
        .sort_values("n", ascending=False).head(30))
display(pd.read_csv(BUILT / "unmapped_terms.csv").head(30))

## Diagnoses and deaths

In [ ]:
d, dr = t["diagnoses"], t["deaths"]
display(d["care_level"].value_counts())
display(d.loc[d["diag_pos"].str.lower().isin(["hdia"]), "icd10_3"].value_counts().head(20))
display(dr["ULORSAK"].str[:3].value_counts().head(20))

In [ ]:
# 30-day mortality after an ED visit, per hospital
day = ed["ed_arrival_datetime"].dt.tz_convert("Europe/Stockholm").dt.tz_localize(None).dt.normalize()
death = pd.to_datetime(ed["lopnr_swed"].map(dr.set_index("lopnr_swed")["date_death_minimp"]))
days = (death - day).dt.days
print("visits with death date before arrival:", (days < 0).sum())
display(ed.assign(died30=days.between(0, 30)).groupby("ed_hospital")["died30"].agg(["size", "mean"]).round(4))